In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",   # or gemini-2.0-pro, gemini-2.5-pro, etc.
    temperature=0.2
)

In [4]:
class BlogState(TypedDict):
    title : str
    outline : str
    content : str

In [6]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [7]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [9]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [10]:
intial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(intial_state)

print(final_state)

{'title': 'Rise of AI in India', 'outline': 'Here\'s a detailed outline for a blog post on "The Rise of AI in India," designed to be engaging, informative, and well-structured.\n\n---\n\n## Blog Title Options:\n\n*   **India\'s AI Ascent: Charting the Rise of Artificial Intelligence in the Subcontinent**\n*   **From Silicon Valley to Silicon Bharat: India\'s Meteoric Rise in the AI Landscape**\n*   **The AI Revolution: How India is Embracing and Shaping the Future of Artificial Intelligence**\n*   **Beyond the Hype: Unpacking India\'s Rapid Growth in the Global AI Arena**\n\n---\n\n## Detailed Blog Outline:\n\n**I. Introduction (Approx. 150-200 words)**\n\n*   **A. Engaging Hook:**\n    *   Start with a broad statement about AI\'s global transformative power.\n    *   Immediately pivot to India\'s unique position and rapid emergence in the AI space.\n    *   *Example:* "Artificial Intelligence is no longer a futuristic concept; it\'s a present-day reality reshaping industries and socie